In [0]:
%pip install sentence-transformers faiss-cpu langchain langchain-community

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

faiss_index_path = "/Volumes/workspace/365pdf/365pdfupload/faiss_intro_tableau_index"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.load_local(
    faiss_index_path,
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS vector index loaded successfully")

In [0]:
question = "What is Tableau used for?"

results_with_scores = vector_store.similarity_search_with_score(
    query=question,
    k=4
)

for i, (doc, score) in enumerate(results_with_scores, start=1):
    print(f"Result {i}")
    print("Score:", score)
    print("Metadata:", doc.metadata)
    print(doc.page_content[:700])
    print("-" * 80)

In [0]:
question = "What is the capital of Japan?"

results_with_scores = vector_store.similarity_search_with_score(
    query=question,
    k=4
)

for i, (doc, score) in enumerate(results_with_scores, start=1):
    print(f"Result {i}")
    print("Score:", score)
    print("Metadata:", doc.metadata)
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
RELEVANCE_THRESHOLD = 1.3

def ask_pdf_with_scores(question, k=4):
    results_with_scores = vector_store.similarity_search_with_score(
        query=question,
        k=k
    )

    if not results_with_scores:
        return "I could not find any relevant information in the uploaded PDF."

    best_doc, best_score = results_with_scores[0]

    print("Question:", question)
    print("Best relevance score:", best_score)
    print()

    if best_score > RELEVANCE_THRESHOLD:
        return """
I could not find a confident answer in the uploaded PDF.

Please ask a question related to the uploaded document.
"""

    context_parts = []
    source_lines = []

    for i, (doc, score) in enumerate(results_with_scores, start=1):
        chunk_id = doc.metadata.get("chunk_id")
        source = doc.metadata.get("source")

        context_parts.append(doc.page_content)
        source_lines.append(
            f"Source {i}: {source}, chunk_id={chunk_id}, score={score:.4f}"
        )

    combined_context = "\n\n".join(context_parts)

    answer = f"""
Answer based only on the uploaded PDF:

{combined_context[:2500]}

Sources:
{chr(10).join(source_lines)}
"""

    return answer

In [0]:
print(ask_pdf_with_scores("What is Tableau used for?"))

In [0]:
print(ask_pdf_with_scores("What are dimensions and measures in Tableau?"))

In [0]:
print(ask_pdf_with_scores("How do you create a visualization in Tableau?"))

In [0]:
print(ask_pdf_with_scores("What is the capital of Japan?"))

In [0]:
RELEVANCE_THRESHOLD = 1.3

def ask_pdf_chatbot(question, k=4):
    results_with_scores = vector_store.similarity_search_with_score(
        query=question,
        k=k
    )

    if not results_with_scores:
        return "I could not find any relevant information in the uploaded PDF."

    best_doc, best_score = results_with_scores[0]

    if best_score > RELEVANCE_THRESHOLD:
        return """
I could not find a confident answer in the uploaded PDF.

Please ask a question related to the uploaded document.
"""

    context_parts = []
    source_lines = []

    for i, (doc, score) in enumerate(results_with_scores, start=1):
        chunk_id = doc.metadata.get("chunk_id")
        source = doc.metadata.get("source")

        context_parts.append(doc.page_content)
        source_lines.append(
            f"- {source}, chunk_id={chunk_id}, score={score:.4f}"
        )

    combined_context = "\n\n".join(context_parts)

    response = f"""
Answer from uploaded PDF:

{combined_context[:1800]}

Sources:
{chr(10).join(source_lines)}
"""

    return response

In [0]:
print(ask_pdf_chatbot("What is Tableau used for?"))

In [0]:
print(ask_pdf_chatbot("What is the capital of Japan?"))

In [0]:
print("PDF Q&A Chatbot is ready.")
print("Ask a question about your uploaded PDF.")
print("Type 'exit' to stop.")
print("-" * 80)

while True:
    question = input("You: ")

    if question.lower().strip() in ["exit", "quit", "stop"]:
        print("Chatbot: Goodbye.")
        break

    answer = ask_pdf_chatbot(question)
    print("\nChatbot:")
    print(answer)
    print("-" * 80)